In [49]:
import numpy as np
import matplotlib.pyplot as plt
import time

plt.rcParams.update({'figure.dpi': 100, 'axes.grid': True, 'grid.alpha': 0.3,
                     'font.size': 11, 'figure.figsize': (10, 5)})
np.set_printoptions(precision=4, suppress=True)

In [50]:
def _sym(A):
    return 0.5 * (A + A.T)

def chol_inv(A, jitter=1e-9):
    A = _sym(np.asarray(A, dtype=float))
    n = A.shape[0]; I = np.eye(n); eps = jitter
    for _ in range(8):
        try:
            L = np.linalg.cholesky(A + eps * I)
            return np.linalg.solve(L.T, np.linalg.solve(L, I))
        except np.linalg.LinAlgError:
            eps *= 10.0
    return np.linalg.solve(A + eps * I, I)

def chol_solve(A, B, jitter=1e-9):
    A = _sym(np.asarray(A, dtype=float))
    B = np.asarray(B, dtype=float)
    n = A.shape[0]; I = np.eye(n); eps = jitter
    for _ in range(8):
        try:
            L = np.linalg.cholesky(A + eps * I)
            return np.linalg.solve(L.T, np.linalg.solve(L, B))
        except np.linalg.LinAlgError:
            eps *= 10.0
    raise np.linalg.LinAlgError("chol_solve failed")

def lu_decomp(A):
    A = np.asarray(A, dtype=float)
    n = A.shape[0]
    L = np.eye(n); U = A.copy()
    for i in range(n):
        for j in range(i + 1, n):
            if U[i, i] == 0:
                raise ValueError("Matrix is singular.")
            L[j, i] = U[j, i] / U[i, i]
            U[j] -= L[j, i] * U[i]
    return L, U

def lu_solve(L, U, b):
    L = np.asarray(L, dtype=float)
    U = np.asarray(U, dtype=float)
    b = np.asarray(b, dtype=float)
    n = L.shape[0]
    y = np.zeros(n)
    for i in range(n):
        y[i] = b[i] - np.dot(L[i, :i], y[:i])
    x = np.zeros(n)
    for i in reversed(range(n)):
        if U[i, i] == 0:
            raise ValueError("Matrix is singular.")
        x[i] = (y[i] - np.dot(U[i, i + 1:], x[i + 1:])) / U[i, i]
    return x

def rollout(F, x0, U):
    U = np.asarray(U, dtype=float)
    if U.ndim == 1: U = U.reshape(-1, 1)
    N_steps, n = U.shape[0], x0.size
    X = np.zeros((N_steps + 1, n)); X[0] = x0
    for k in range(N_steps):
        X[k + 1] = F(X[k], U[k])
    return X

def as_terminal_weight(alpha, n):
    A = np.asarray(alpha, dtype=float)
    if A.ndim == 0: return float(A) * np.eye(n)
    if A.ndim == 1: return np.diag(A)
    return _sym(A)

def _wrap(e, wrap_idx):
    if not wrap_idx: return e
    e = np.asarray(e, dtype=float).copy()
    for i in wrap_idx:
        e[i] = (e[i] + np.pi) % (2*np.pi) - np.pi
    return e


print("Utilities defined.")

Utilities defined.


In [51]:
def bruteforce_horizon_search(A_list, B_list, Q, R, Qf, w, x0, T_max):
    n, m = x0.size, R.shape[0]
    J_curve = np.zeros(T_max)
    for T in range(1, T_max + 1):
        # Riccati initialization
        P = Qf.copy()                          
        for k in range(T - 1, -1, -1):         #Ricatti recursion
            A, B = A_list[k], B_list[k]
            S = R + B.T @ P @ B
            K = np.linalg.solve(S, B.T @ P @ A)
            P = _sym(Q + A.T @ P @ A - A.T @ P @ B @ K)
        J_curve[T - 1] = 0.5 * x0 @ P @ x0 + w * T   # Cost for every horizon length T
    return J_curve

print("bruteforce_horizon_search defined.")

bruteforce_horizon_search defined.


In [52]:
def hop_lqr_ltv(A_list, B_list, Q, R, Qf, w, x0, T_max):
    # hop_lqr generalized to a per-stage Q_k (LTV): Q is now a list, indexed
    # per k, everywhere it appears below. Still no augmentation needed (cost
    # is homogeneous quadratic in x, regulation to the origin), and still
    # requires every Q_k, Qf to be PD (Assumption 2) -- unlike
    # shifted_hop_lqr_ltv. Deliberately uses np.linalg.inv, not chol_inv:
    # chol_inv silently escalates its jitter when a matrix isn't PD, so if
    # you hand-edit Q to be singular/indefinite this should fail loudly or
    # give a visibly WRONG T*, not get quietly patched over.
    #
    # Same Theorem-1 composed-map idea as bruteforce_horizon_search_ltv, but
    # the backward Riccati sweep is run ONCE (forward, as an inverse/"E,F,G"
    # map composition) instead of once per horizon T: O(n^3 T_max) total
    # instead of O(n^3 T_max^2).
    R_inv = np.linalg.inv(R)

    E_list, F_list, G_list = [], [], []
    for k in range(T_max):
        A, B = A_list[k], B_list[k]
        Ek = np.linalg.inv(Q[k])               # local inverse-cost block
        Fk = Ek @ A.T
        Gk = A @ Ek @ A.T + B @ R_inv @ B.T
        E_list.append(Ek); F_list.append(Fk); G_list.append(_sym(Gk))

    # Phase 1: compose the per-stage maps forward, independent of any
    # terminal condition.
    Ebar = [E_list[0].copy()]
    Fbar = [F_list[0].copy()]
    Gbar = [G_list[0].copy()]
    for k in range(1, T_max):
        W = np.linalg.inv(E_list[k] + Gbar[-1])
        Ebar.append(_sym(Ebar[-1] - Fbar[-1] @ W @ Fbar[-1].T))
        Fbar.append(Fbar[-1] @ W @ F_list[k])
        Gbar.append(_sym(G_list[k] - F_list[k].T @ W @ F_list[k]))

    # Phase 2: combine with the (fixed) terminal weight Qf at every horizon.
    Xt = np.linalg.inv(Qf)
    J_curve = np.zeros(T_max)
    for t in range(1, T_max + 1):
        Wt = np.linalg.inv(Xt + Gbar[t - 1])
        X0 = _sym(Ebar[t - 1] - Fbar[t - 1] @ Wt @ Fbar[t - 1].T)
        P0 = np.linalg.inv(X0)
        J_curve[t - 1] = 0.5 * x0 @ P0 @ x0 + w * t
    return J_curve

print("hop_lqr_ltv defined.")

hop_lqr_ltv defined.


In [53]:
def shifted_hop_lqr_ltv(A_list, B_list, Q, R, Qf, w, x0, c, T_max):
    """Solve the shifted HOP LQR problem (Algorithm 1).

    Same O(n^3 T_max) composed-map idea as hop_lqr, but tracks
    Phat_k = (P_k + cI)^-1 instead of Ptilde_k = P_k^-1. Phat stays invertible
    even where the true cost-to-go P_k is singular or indefinite, as long as c
    is large enough that Q_k + cI and Qf + cI are PD everywhere -- that's what
    removes HOP's Assumption 2 (Q_k > 0). The shift is undone only once, at
    the very end of each horizon query (line 17), not at every stage.

    chol_inv is deliberately NOT used here: it silently assumes its input is
    (near) PD and, when it isn't, keeps escalating its diagonal jitter up to
    ~0.1 before giving up -- for the matrices below that's not "a little
    regularization", it silently returns the inverse of a badly different
    matrix with no warning. Several of the intermediate matrices here (W_k,
    and Phat_T + Gbar) are genuinely indefinite even though Q_k + cI itself is
    PD by construction -- that's expected (Remark 5: "LDL^T or LU -- not
    Cholesky"), not a bug, so every inverse below uses plain
    np.linalg.inv/solve instead.
    """
    n = x0.size
    I = np.eye(n)
    R_inv = np.linalg.inv(R)

    # ---- Phase 1a: single-step coefficients (lines 2-8) ----
    Ep, Fp, Gp = [], [], []
    for k in range(T_max):
        A, B = A_list[k], B_list[k]
        Qbar = Q[k] + c * I                                   # line 3: Q~_k
        Et = np.linalg.inv(Qbar)                              # line 4: E~_k (PD by choice of c)
        Ft = Et @ A.T                                         # line 4: F~_k
        Gt = _sym(A @ Et @ A.T + B @ R_inv @ B.T)             # line 4: G~_k

        M = I - c * Gt        # line 5: I - cG~_k is symmetric but INDEFINITE
                               # in the normal operating regime -- solve
                               # against it directly, never via Cholesky.
        Fk = np.linalg.solve(M.T, Ft.T).T                     # line 6: F'_k
        Gk = _sym(np.linalg.solve(M.T, Gt.T).T)               # line 6: G'_k
        Ek = _sym(Et + c * Fk @ Ft.T)                         # line 7: E'_k
        Ep.append(Ek); Fp.append(Fk); Gp.append(Gk)

    # ---- Phase 1b: compose forward, independent of any terminal condition
    # (lines 9-13, HOP Eq. 14, unchanged) ----
    Ebar = [Ep[0].copy()]; Fbar = [Fp[0].copy()]; Gbar = [Gp[0].copy()]
    for k in range(1, T_max):
        W = np.linalg.inv(Ep[k] + Gbar[-1])                   # line 11: W_k (not PD in general)
        Ebar.append(_sym(Ebar[-1] - Fbar[-1] @ W @ Fbar[-1].T))   # line 12: Ebar_k
        Fbar.append(Fbar[-1] @ W @ Fp[k])                         # line 12: Fbar_k
        Gbar.append(_sym(Gp[k] - Fp[k].T @ W @ Fp[k]))            # line 12: Gbar_k

    # ---- Phase 2: horizon queries against the fixed terminal weight Qf,
    # unchanged (lines 14-18) ----
    PhatT = np.linalg.inv(Qf + c * I)                         # line 15: Phat_T (PD by choice of c)
    J_curve = np.zeros(T_max)
    for t in range(1, T_max + 1):
        Phat0 = _sym(Ebar[t - 1]
                     - Fbar[t - 1] @ np.linalg.inv(PhatT + Gbar[t - 1]) @ Fbar[t - 1].T)  # line 16
        P0 = _sym(np.linalg.inv(Phat0)) - c * I               # line 17: undo the shift
        J_curve[t - 1] = 0.5 * x0 @ P0 @ x0 + w * t
    return J_curve

print("shifted_hop_lqr (Algorithm 1) defined.")

shifted_hop_lqr (Algorithm 1) defined.


In [54]:
import numpy as np

rng = np.random.default_rng(0)
n, m, T_max = 4, 2, 40

A_list = [np.eye(n) + 0.1 * rng.normal(size=(n, n)) for _ in range(T_max)]
B_list = [rng.normal(size=(n, m)) for _ in range(T_max)]

R = rng.normal(size=(m, m)); R = _sym(R @ R.T + np.eye(m))
Qf = rng.normal(size=(n, n)); Qf = _sym(Qf @ Qf.T + 2 * np.eye(n))
x0 = rng.normal(size=n)

# One independent PD matrix per stage -- hand-edit entries of Q_list below
# (e.g. Q_list[k] -= 5*np.eye(n), or set an eigenvalue negative) to make
# individual stages singular/indefinite and see how hop_lqr vs
# shifted_hop_lqr (with c = choose_shift(Q_list, [Qf]*T_max)) respond.
Q_list = []
for _ in range(T_max):
    M = rng.normal(size=(n, n))
    Q_list.append(_sym(M @ M.T + np.eye(n)))

print(f"n={n} m={m} T_max={T_max}")
print("min eig per stage Q_k:", [round(float(np.linalg.eigvalsh(Q_list[k])[0]), 3) for k in range(T_max)])
print("min eig Qf:", float(np.linalg.eigvalsh(Qf)[0]))
print("min eig R:", float(np.linalg.eigvalsh(R)[0]))


n=4 m=2 T_max=40
min eig per stage Q_k: [1.082, 1.002, 1.223, 1.038, 1.0, 1.031, 1.054, 1.045, 1.0, 1.052, 1.06, 1.03, 1.109, 1.009, 1.029, 1.038, 1.525, 1.242, 1.093, 1.116, 1.211, 1.597, 1.613, 1.313, 1.136, 1.275, 1.108, 1.049, 1.038, 1.2, 1.003, 1.0, 1.003, 1.045, 1.085, 1.001, 1.826, 1.021, 1.019, 1.011]
min eig Qf: 2.0007369231693546
min eig R: 1.1956991334970941


In [55]:
import numpy as np

# System dimensions
nx = 4
nu = 2

# Time grid
N = 100
T = 10.0
t = np.linspace(0, T, N)

def A(t):
    return np.array([
        [0.0,       1.0,          0.0,       0.0],
        [-2.0,      -0.2,         5*np.sin(t), 0.0],
        [0.0,        0.0,          0.0,       1.0],
        [0.0,        0.0,         -1.0,      -0.3]
    ])

def B(t):
    return np.array([
        [0.0,       0.0],
        [1.0,       0.1*np.cos(t)],
        [0.0,       0.0],
        [0.2,       1.0]
    ])

In [56]:
dt = t[1] - t[0]
A_list = np.array([np.eye(nx) + dt*A(tk) for tk in t])
B_list = np.array([dt*B(tk) for tk in t])   # dt = t[1]-t[0]

In [57]:
A_list.shape
# (100, 4, 4)

B_list.shape
# (100, 4, 2)

(100, 4, 2)

In [58]:
def bruteforce_horizon_search_ltv(A_list, B_list, Q_list, R, Qf, w, x0, T_max):
    """bruteforce_horizon_search generalized to a per-stage Q_k (LTV). No
    inverse of Q_k needed anywhere, so this stays the reliable ground truth
    however Q_list gets hand-edited (singular, indefinite, whatever) --
    compare hop_lqr_ltv / shifted_hop_lqr_ltv against this."""
    J_curve = np.zeros(T_max)
    for T in range(1, T_max + 1):
        P = Qf.copy()
        for k in range(T - 1, -1, -1):
            A, B = A_list[k], B_list[k]
            S = R + B.T @ P @ B
            K = np.linalg.solve(S, B.T @ P @ A)
            P = _sym(Q_list[k] + A.T @ P @ A - A.T @ P @ B @ K)
        J_curve[T - 1] = 0.5 * x0 @ P @ x0 + w * T
    return J_curve

# One independent PD matrix per stage, matching A_list/B_list's N=100 steps
# above (nx=4) -- hand-edit Q_list[k] afterward (e.g. Q_list[k] -= 5*np.eye(nx),
# or flip an eigenvalue negative) to check hop_lqr_ltv / shifted_hop_lqr_ltv
# against bruteforce_horizon_search_ltv, which stays correct regardless.
Q_list = []
for _ in range(N):
    M = rng.normal(size=(nx, nx))
    Q_list.append(_sym(M @ M.T + np.eye(nx)))

J_bf = bruteforce_horizon_search_ltv(A_list, B_list, Q_list, R, Qf, 0.2, x0, N)
print("bruteforce_horizon_search_ltv:  T* =", int(np.argmin(J_bf)) + 1, " J* =", float(J_bf.min()))

bruteforce_horizon_search_ltv:  T* = 1  J* = 60.28172405346991


In [59]:
for k in range(N):
    U, s, _ = np.linalg.svd(Q_list[k]); s[0] = 0.0
    Q_list[k] = _sym(U @ np.diag(s) @ U.T)

In [62]:
try:
      J_hop = hop_lqr_ltv(A_list, B_list, Q_list, R, Qf, 0.2, x0, N)
except Exception as e:
      print("Error in hop_lqr_ltv:", e)
      J_hop = None

if J_hop is not None:
      relerr_hop = float(np.max(np.abs(J_hop - J_bf) / np.abs(J_bf)))
      print("hop_lqr_ltv:                    T* =", int(np.argmin(J_hop)) + 1,
            " J* =", float(J_hop.min()), " max relerr vs bruteforce:", relerr_hop)

Error in hop_lqr_ltv: Singular matrix


In [61]:
def choose_shift(Q_list, QT_list, margin=2.0, floor=1e-3):
    """c must exceed -lambda_min over every stage/terminal cost matrix so
    Q_k + cI, Q_T + cI are PD everywhere -- Algorithm 1 line 1's c_0, picked
    automatically instead of by hand."""
    lo = 0.0
    for M in list(Q_list) + list(QT_list):
        lo = min(lo, float(np.linalg.eigvalsh(_sym(M))[0]))
    return max(margin * (-lo), floor)

c = choose_shift(Q_list, [Qf] * N)
J_shift = shifted_hop_lqr_ltv(A_list, B_list, Q_list, R, Qf, 0.2, x0, c, N)
relerr_shift = float(np.max(np.abs(J_shift - J_bf) / np.abs(J_bf)))
print(f"shifted_hop_lqr_ltv (c={c:.4g}):  T* =", int(np.argmin(J_shift)) + 1,
      " J* =", float(J_shift.min()), " max relerr vs bruteforce:", relerr_shift)

shifted_hop_lqr_ltv (c=0.001):  T* = 1  J* = 45.95972887199186  max relerr vs bruteforce: 0.44088980080709855


# DDP without Horizon Optimization


In [66]:
from dataclasses import dataclass, field
import numpy as np
import jax
import jax.numpy as jnp
jax.config.update("jax_enable_x64", True)

In [68]:
@dataclass
class Cartpole:
    M: float = 1.0
    m: float = 0.2
    l: float = 0.5
    g: float = 9.81
    b_p: float = 0.0
    b_th: float = 0.0
    dt: float = 0.05
    nx: int = 4
    nu: int = 1

    def _ct(self, x, u, np_):
        """Continuous dynamics.  theta from UPRIGHT."""
        p, th, pd, thd = x[0], x[1], x[2], x[3]
        f = u[0]
        s, c = np_.sin(th), np_.cos(th)
        denom = self.M + self.m * s ** 2
        pdd = (f + self.m * s * (self.l * thd ** 2 + self.g * c)) / denom - self.b_p * pd
        thdd = (-f * c - self.m * self.l * thd ** 2 * c * s
                - (self.M + self.m) * self.g * s) / (self.l * denom) - self.b_th * thd
        return np_.stack([pd, thd, pdd, thdd])

    def _rk4(self, x, u, np_):
        h = self.dt
        f1 = self._ct(x, u, np_)
        f2 = self._ct(x + 0.5 * h * f1, u, np_)
        f3 = self._ct(x + 0.5 * h * f2, u, np_)
        f4 = self._ct(x + h * f3, u, np_)
        return x + (h/6.0) * (f1 + 2 * f2 + 2 * f3 + f4)

    def _step(self, x, u):
        x = np.asarray(x, dtype=float).reshape(-1)
        u = np.asarray(x, dtype=float).reshape(-1)
        return self._rk4(x, u, np)

    def _step_jax(self, x, u):
        return self._rk4(x, u, jnp)
    


In [69]:
class Linearizer:
    def __init__(self, sys):
        self.sys = sys

        self._A = jax.jit(jax.vmap(jax.jacfwd(sys._step_jax, argnums=0)))
        self._B = jax.jit(jax.vmap(jax.jacfwd(sys._step_jax, argnums=1)))

    def __call__(self, X, U):
        Xk = np.asarray(X[:-1], dtype=float)
        Uk = np.asarray(U, dtype=float)

        return (np.asarray(self._A(jnp.asarray(Xk), jnp.asarray(Uk))),
                np.asarray(self._B(jnp.asarray(Xk), jnp.asarray(Uk))))


### Analytical Cost derivatives

In [ ]:
@dataclass
class SwingupCost:
    theta_goal: float = np.pi
    
    